# Entrenamiento y comparación (issue #10)
Wrapper delgado: llama al mismo código del backend (`adapters.models`). No duplica preparación ni reescribe el CSV. Metodología del ticket: CV estratificada solo en train (5 pedidos), GridSearchCV chica, contraste en validación, test un solo uso.


In [ ]:
import sys
from pathlib import Path

CANDIDATOS = [Path.cwd() / "backend" / "src", Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "collection").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import collection
print("src:", SRC.exists(), "schema:", collection.SCHEMA_VERSION)


In [ ]:
ROOTS = [Path.cwd(), Path.cwd().parent]
DATASET = next(r / "dataset" / "salida" / "tramos.csv" for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
print("dataset:", DATASET)


In [ ]:
from collection.adapters.datasets.ml_preparation import prepare_dataset

prepared = prepare_dataset(DATASET, seed=0)
print("sizes:", {k: v.metadata["size"] for k, v in prepared.partitions.items()})
print("dist:", prepared.metadata["class_distribution"])
assert sum(v.metadata["size"] for v in prepared.partitions.values()) == len(prepared.enriched) == 297
assert not hasattr(prepared.preprocessor, "transformers_")


In [ ]:
from collection.adapters.models.sklearn_evaluator import SklearnEvaluator
from collection.application.models.evaluation_protocol import EvaluationProtocol

evaluator = SklearnEvaluator()
comparison, rows = evaluator.evaluate_preparation(prepared, ["arbol", "logistica"], EvaluationProtocol())
for m in comparison.metrics:
    print(f"{m.model_id}: cv_f1={m.cv_f1_macro_mean:.4f} val_f1={m.validation_f1_macro:.4f} best={m.best_params}")
print("pedidos/usados:", comparison.record["cv_folds_pedidos"], comparison.record["cv_folds_usados"])
print("elegido:", comparison.chosen_model_id)
assert comparison.record["cv_folds_pedidos"] == 5 and comparison.record["cv_folds_usados"] == 5
assert all(m.test_accuracy is None for m in comparison.metrics)
best = max(comparison.metrics, key=lambda m: (m.validation_f1_macro, m.validation_accuracy))
assert comparison.chosen_model_id == best.model_id


In [ ]:
final = evaluator.evaluate_final_on_test(comparison, rows, EvaluationProtocol())
chosen = next(m for m in final.metrics if m.model_id == final.chosen_model_id)
print(f"test un solo uso ({chosen.model_id}): acc={chosen.test_accuracy:.4f} f1={chosen.test_f1_macro:.4f}")
assert chosen.test_accuracy is not None and chosen.test_f1_macro is not None
assert sum(1 for m in final.metrics if m.test_accuracy is not None) == 1
print("NOTA: metricas = ajuste a la regla sintetica, no desempeno operativo.")
